In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/spaceship-titanic/sample_submission.csv
/kaggle/input/competitions/spaceship-titanic/train.csv
/kaggle/input/competitions/spaceship-titanic/test.csv


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

train_df = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')
test_df  = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/test.csv')

test_passenger_ids = test_df['PassengerId'].copy()

y = train_df['Transported'].astype(int)
train_df = train_df.drop('Transported', axis=1)

all_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)

expense_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

all_df.loc[all_df['CryoSleep'] == True, expense_cols] = 0.0

mask = all_df['VIP'] == False
all_df.loc[mask, 'HomePlanet'] = all_df.loc[mask, 'HomePlanet'].fillna('Earth')
mask = all_df['VIP'] == True
all_df.loc[mask, 'HomePlanet'] = all_df.loc[mask, 'HomePlanet'].fillna('Europa')

for col in all_df.columns:
    if all_df[col].isnull().sum() == 0:
        continue
    if all_df[col].dtype == object or all_df[col].dtype == bool:
        all_df[col] = all_df[col].fillna(all_df[col].mode()[0])
    else:
        all_df[col] = all_df[col].fillna(all_df[col].median())

print("剩余缺失值:", all_df.isnull().sum().sum())  


cabin_split = all_df['Cabin'].str.split('/', expand=True)
all_df['Deck']     = cabin_split[0]
all_df['CabinNum'] = cabin_split[1]
all_df['Side']     = cabin_split[2]
all_df = all_df.drop('Cabin', axis=1)

all_df['Group']     = all_df['PassengerId'].str.split('_').str[0]
all_df['GroupSize'] = all_df.groupby('Group')['Group'].transform('count')

all_df['TotalSpent'] = all_df[expense_cols].sum(axis=1)
all_df = all_df.drop(['PassengerId', 'Name', 'Group'], axis=1)

for col in all_df.columns:
    if all_df[col].dtype == object or all_df[col].dtype == bool:
        all_df[col] = all_df[col].astype(str)

all_df['CabinNum'] = pd.to_numeric(all_df['CabinNum'], errors='coerce')
all_df['CabinNum'] = all_df['CabinNum'].fillna(all_df['CabinNum'].median())

for col in all_df.columns:
    if all_df[col].dtype == object:
        le = LabelEncoder()
        all_df[col] = le.fit_transform(all_df[col])

X = all_df.iloc[:len(train_df), :].copy()
X_test = all_df.iloc[len(train_df):, :].copy()

print("Train shape:", X.shape)
print("Test shape:", X_test.shape)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
print("accuracy score:", accuracy_score(y_val, y_pred))

test_pred = model.predict(X_test)
submission = pd.DataFrame({
    'PassengerId': test_passenger_ids,
    'Transported': test_pred.astype(bool)
})
submission.to_csv('submission.csv', index=False)
print(submission.head())
print("Submission saved", len(submission), "lines")

剩余缺失值: 0
Train shape: (8693, 15)
Test shape: (4277, 15)
accuracy score: 0.7872340425531915
  PassengerId  Transported
0     0013_01         True
1     0018_01        False
2     0019_01         True
3     0021_01         True
4     0023_01        False
Submission saved 4277 lines
